In [ ]:
# CELL 1: Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

Mounted at /content/drive
Drive mounted.


In [ ]:
# CELL 2: Unzip — skips if already extracted
import os, zipfile

zip_path     = '/content/drive/MyDrive/AVEC2014.zip'
extract_path = '/content/avec2014'
marker       = os.path.join(extract_path, 'AVEC2014', 'labels.csv')

if os.path.exists(marker):
    print('Already extracted — skipping.')
else:
    print('Extracting...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)
    print('Done.')

Extracting...
Done.


In [ ]:
# CELL 3: Imports and constants
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import subprocess
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print(subprocess.getoutput('nvidia-smi | grep "GPU 0"'))

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

DATA_ROOT     = '/content/avec2014/AVEC2014'
FRAME_ROOT    = '/content/avec2014_frames'
BASELINE_CKPT = '/content/drive/MyDrive/c3d_avec2014_best.pth'
QR_CKPT       = '/content/drive/MyDrive/quantile_checkpoint_epoch7.pth'
GENDER_CSV    = '/content/drive/MyDrive/avec2014_gender.csv'
FUQ_RESULTS   = '/content/drive/MyDrive/avec2014_fuq_results.csv'

CLIP_LEN    = 16
STRIDE      = 8
BATCH_SIZE  = 32   # A100: 32 | L4: 8 | T4: 4
N_QUANTILES = 99
M_BINS      = 4
ALPHA       = 0.1

print(f'BASELINE_CKPT exists: {os.path.exists(BASELINE_CKPT)}')
print(f'QR_CKPT       exists: {os.path.exists(QR_CKPT)}')
print(f'GENDER_CSV    exists: {os.path.exists(GENDER_CSV)}')
print(f'FUQ_RESULTS   exists: {os.path.exists(FUQ_RESULTS)}')

Device: cuda

BASELINE_CKPT exists: True
QR_CKPT       exists: True
GENDER_CSV    exists: True
FUQ_RESULTS   exists: True


In [ ]:
# CELL 4: Keep-alive
import time, threading

def keep_alive():
    while True:
        time.sleep(60)
        print('.', end='', flush=True)

t = threading.Thread(target=keep_alive, daemon=True)
t.start()
print('Keep-alive started.')

Keep-alive started.


In [ ]:
# CELL 5: Load labels and gender/age map
def load_labels():
    df = pd.read_csv(os.path.join(DATA_ROOT, 'labels.csv'))
    labels = {}
    for _, row in df.iterrows():
        key = str(row['filename']).strip().replace('\\', '/')
        key = os.path.splitext(key)[0]
        labels[key] = float(row['BDI-II'])
    print(f'Labels loaded: {len(labels)}')
    return labels

def load_gender_map():
    if os.path.exists(GENDER_CSV):
        df = pd.read_csv(GENDER_CSV)
        gmap = {}
        amap = {}
        for _, row in df.iterrows():
            key = str(row['filename']).strip().replace('\\', '/')
            key = os.path.splitext(key)[0]
            gmap[key] = str(row['gender']).strip().upper()
            amap[key] = str(row['age_group']).strip()
        print(f'Gender/age map loaded: {len(gmap)} entries')
        return gmap, amap
    print('No gender.csv found.')
    return {}, {}

labels               = load_labels()
gender_map, age_map  = load_gender_map()

Labels loaded: 300
Gender/age map loaded: 300 entries


In [ ]:
# CELL 6: Collect videos
def collect_videos(folders, labels):
    if isinstance(folders, str):
        folders = [folders]
    items = []
    for folder in folders:
        if not os.path.exists(folder):
            print(f'WARNING: not found: {folder}')
            continue
        for root, _, files in os.walk(folder):
            for f in sorted(files):
                if not f.lower().endswith('.mp4'):
                    continue
                path = os.path.join(root, f)
                rel  = os.path.relpath(path, DATA_ROOT).replace('\\', '/')
                stem = os.path.splitext(rel)[0]
                if stem in labels:
                    items.append((path, stem, labels[stem]))
    return items

TRAIN_DIRS = [
    os.path.join(DATA_ROOT, 'Training'),
    os.path.join(DATA_ROOT, 'Development'),
]
CAL_DIR  = os.path.join(DATA_ROOT, 'Testing', 'Northwind')
TEST_DIR = os.path.join(DATA_ROOT, 'Testing', 'Freeform')

train_items = collect_videos(TRAIN_DIRS, labels)
cal_items   = collect_videos(CAL_DIR,    labels)
test_items  = collect_videos(TEST_DIR,   labels)

print(f'Train : {len(train_items)}  Cal : {len(cal_items)}  Test : {len(test_items)}')

train_labels_arr = np.array([l for _, _, l in train_items])
LABEL_MEAN = float(train_labels_arr.mean())
LABEL_STD  = float(train_labels_arr.std())
print(f'Label norm — mean: {LABEL_MEAN:.2f}  std: {LABEL_STD:.2f}')

Train : 200  Cal : 50  Test : 50
Label norm — mean: 15.34  std: 12.07


In [ ]:
# CELL 7: Pre-extract frames (skips if already done)
os.makedirs(FRAME_ROOT, exist_ok=True)

def extract_video(path, stem):
    out_dir = os.path.join(FRAME_ROOT, stem.replace('/', '_'))
    if os.path.exists(out_dir) and len(os.listdir(out_dir)) >= CLIP_LEN:
        return
    os.makedirs(out_dir, exist_ok=True)
    cap = cv2.VideoCapture(path)
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (112, 112))
        cv2.imwrite(
            os.path.join(out_dir, f'{idx:05d}.jpg'),
            frame,
            [cv2.IMWRITE_JPEG_QUALITY, 95]
        )
        idx += 1
    cap.release()

all_items = train_items + cal_items + test_items
print(f'Extracting frames for {len(all_items)} videos...')
for path, stem, label in tqdm(all_items):
    extract_video(path, stem)
print('Extraction complete.')

Extracting frames for 300 videos...


 15%|█▌        | 45/300 [00:36<03:30,  1.21it/s]

.

 33%|███▎      | 98/300 [01:36<05:37,  1.67s/it]

.

...

.

 94%|█████████▍| 282/300 [04:37<00:22,  1.27s/it]

.

100%|██████████| 300/300 [05:08<00:00,  1.03s/it]

Extraction complete.


In [ ]:
# CELL 8: Dataset classes
def load_frames_from_ssd(stem, start, n=CLIP_LEN):
    out_dir = os.path.join(FRAME_ROOT, stem.replace('/', '_'))
    frames  = []
    for i in range(start, start + n):
        fpath = os.path.join(out_dir, f'{i:05d}.jpg')
        if not os.path.exists(fpath):
            break
        frame = cv2.imread(fpath)
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    return frames

def count_frames(stem):
    out_dir = os.path.join(FRAME_ROOT, stem.replace('/', '_'))
    if not os.path.exists(out_dir):
        return 0
    return len([f for f in os.listdir(out_dir) if f.endswith('.jpg')])

def frames_to_tensor(frames):
    arr = np.stack(frames, axis=0).astype(np.float32) / 255.0
    arr = (arr - MEAN) / STD
    arr = arr.transpose(3, 0, 1, 2)
    return torch.from_numpy(arr)

def get_clip_starts(T, stride=STRIDE):
    return list(range(0, T - CLIP_LEN + 1, stride))

class AVEC2014Train(Dataset):
    def __init__(self, items):
        self.samples = []
        for path, stem, label in items:
            T = count_frames(stem)
            if T < CLIP_LEN:
                continue
            for s in get_clip_starts(T):
                self.samples.append((stem, s, label))
        print(f'Training clips: {len(self.samples)}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        stem, start, label = self.samples[idx]
        frames = load_frames_from_ssd(stem, start, CLIP_LEN)
        if len(frames) < CLIP_LEN:
            while len(frames) < CLIP_LEN:
                frames.append(frames[-1])
        norm_label = (label - LABEL_MEAN) / LABEL_STD
        return frames_to_tensor(frames), torch.tensor(norm_label, dtype=torch.float32)

class AVEC2014Eval(Dataset):
    def __init__(self, items):
        self.samples = []
        for path, stem, label in items:
            T = count_frames(stem)
            if T >= CLIP_LEN:
                self.samples.append((stem, label))
        print(f'Eval videos: {len(self.samples)}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        stem, label = self.samples[idx]
        T     = count_frames(stem)
        clips = []
        for s in get_clip_starts(T, stride=8):
            frames = load_frames_from_ssd(stem, s, CLIP_LEN)
            if len(frames) < CLIP_LEN:
                continue
            clips.append(frames_to_tensor(frames))
        if not clips:
            clips.append(torch.zeros(3, CLIP_LEN, 112, 112))
        return torch.stack(clips), torch.tensor(label, dtype=torch.float32), stem

In [ ]:
# CELL 9: C3D Quantile Model
class C3DQuantile(nn.Module):
    def __init__(self, n_quantiles=99, dropout=0.5):
        super().__init__()
        self.conv1  = nn.Conv3d(3,   64,  kernel_size=(3,3,3), padding=(1,1,1))
        self.pool1  = nn.MaxPool3d(kernel_size=(1,2,2), stride=(1,2,2))
        self.conv2  = nn.Conv3d(64,  128, kernel_size=(3,3,3), padding=(1,1,1))
        self.pool2  = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2))
        self.conv3a = nn.Conv3d(128, 256, kernel_size=(3,3,3), padding=(1,1,1))
        self.conv3b = nn.Conv3d(256, 256, kernel_size=(3,3,3), padding=(1,1,1))
        self.pool3  = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2))
        self.conv4a = nn.Conv3d(256, 512, kernel_size=(3,3,3), padding=(1,1,1))
        self.conv4b = nn.Conv3d(512, 512, kernel_size=(3,3,3), padding=(1,1,1))
        self.pool4  = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2))
        self.conv5a = nn.Conv3d(512, 512, kernel_size=(3,3,3), padding=(1,1,1))
        self.conv5b = nn.Conv3d(512, 512, kernel_size=(3,3,3), padding=(1,1,1))
        self.pool5  = nn.MaxPool3d(kernel_size=(2,2,2), stride=(2,2,2), padding=(0,1,1))
        self.relu    = nn.ReLU(inplace=True)
        self.fc6     = nn.Linear(8192, 4096)
        self.fc7     = nn.Linear(4096, 64)
        self.fc8     = nn.Linear(64, n_quantiles)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.relu(self.conv1(x));  x = self.pool1(x)
        x = self.relu(self.conv2(x));  x = self.pool2(x)
        x = self.relu(self.conv3a(x))
        x = self.relu(self.conv3b(x)); x = self.pool3(x)
        x = self.relu(self.conv4a(x))
        x = self.relu(self.conv4b(x)); x = self.pool4(x)
        x = self.relu(self.conv5a(x))
        x = self.relu(self.conv5b(x)); x = self.pool5(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(self.relu(self.fc6(x)))
        x = self.dropout(self.relu(self.fc7(x)))
        return self.fc8(x)

qr_model = C3DQuantile(n_quantiles=N_QUANTILES).to(device)
print(f'Parameters: {sum(p.numel() for p in qr_model.parameters()):,}')
with torch.no_grad():
    out = qr_model(torch.zeros(2, 3, 16, 112, 112).to(device))
    print(f'Forward pass OK — output: {out.shape}')

Parameters: 61,483,107
Forward pass OK — output: torch.Size([2, 99])


In [ ]:
# CELL 10: Load quantile checkpoint from Drive
# Skips training entirely — uses saved checkpoint from Kaggle session

qr_model.load_state_dict(torch.load(QR_CKPT, map_location=device))
print(f'Quantile model loaded from: {QR_CKPT}')
print(f'File size: {os.path.getsize(QR_CKPT)/1e6:.1f} MB')

Quantile model loaded from: /content/drive/MyDrive/quantile_checkpoint_epoch7.pth
File size: 245.9 MB


In [ ]:
# CELL 11: Build datasets
cal_dataset  = AVEC2014Eval(cal_items)
test_dataset = AVEC2014Eval(test_items)

Eval videos: 50
Eval videos: 50


In [ ]:
# CELL 12: Evaluation functions
QUANTILES = torch.linspace(0.01, 0.99, N_QUANTILES).to(device)

def get_video_quantiles(dataset, model, chunk_size=32):
    model.eval()
    results = []
    with torch.no_grad():
        for clips, label, stem in dataset:
            all_preds = []
            for i in range(0, len(clips), chunk_size):
                chunk = clips[i:i+chunk_size].to(device)
                preds = model(chunk)
                all_preds.append(preds.cpu())
                torch.cuda.empty_cache()
            q_norm = torch.cat(all_preds).mean(dim=0).numpy()
            q_pred = q_norm * LABEL_STD + LABEL_MEAN
            results.append((stem, label.item(), q_pred))
    return results

def eval_point_prediction(results):
    y_true = np.array([r[1] for r in results])
    y_pred = np.array([r[2][49] for r in results])
    mae    = np.mean(np.abs(y_true - y_pred))
    rmse   = np.sqrt(np.mean((y_true - y_pred)**2))
    return mae, rmse

print('Evaluation functions defined.')

Evaluation functions defined.


In [ ]:
# CELL 13: Get predictions on cal and test sets
print('Running evaluation...')
cal_results  = get_video_quantiles(cal_dataset,  qr_model)
test_results = get_video_quantiles(test_dataset, qr_model)

cal_mae,  cal_rmse  = eval_point_prediction(cal_results)
test_mae, test_rmse = eval_point_prediction(test_results)
print(f'Cal  MAE={cal_mae:.4f}  RMSE={cal_rmse:.4f}')
print(f'Test MAE={test_mae:.4f}  RMSE={test_rmse:.4f}')

Running evaluation...
....Cal  MAE=7.5787  RMSE=9.7113
Test MAE=8.4299  RMSE=10.4761


In [ ]:
# CELL 14: CQR
Q_LO_IDX = int(ALPHA / 2 * N_QUANTILES)
Q_HI_IDX = int((1 - ALPHA / 2) * N_QUANTILES) - 1
print(f'Lower quantile: {QUANTILES[Q_LO_IDX].item():.2f}  Upper: {QUANTILES[Q_HI_IDX].item():.2f}')

cal_scores = np.array([
    max(q[Q_LO_IDX] - y, y - q[Q_HI_IDX])
    for _, y, q in cal_results
])
beta = np.quantile(cal_scores, 1 - ALPHA)
print(f'Global beta = {beta:.4f}')

test_intervals_cqr = [
    (stem, y, q[Q_LO_IDX] - beta, q[Q_HI_IDX] + beta)
    for stem, y, q in test_results
]
picp_cqr = np.mean([lo <= y <= hi for _, y, lo, hi in test_intervals_cqr])
mpiw_cqr = np.mean([hi - lo for _, _, lo, hi in test_intervals_cqr])
print(f'CQR: PICP={picp_cqr:.4f}  MPIW={mpiw_cqr:.4f}')

Lower quantile: 0.05  Upper: 0.94
Global beta = 14.9149
CQR: PICP=0.8800  MPIW=36.5309


In [ ]:
# CELL 15: Group Conditional Conformal Prediction
cal_data = pd.DataFrame([{
    'stem':   stem,
    'y_true': y,
    'y_lo':   q[Q_LO_IDX],
    'y_hi':   q[Q_HI_IDX],
    'r':      max(q[Q_LO_IDX] - y, y - q[Q_HI_IDX]),
    'gender': gender_map.get(stem, '?')
} for stem, y, q in cal_results])

print(cal_data['gender'].value_counts())

cal_sorted = cal_data.sort_values('y_true').reset_index(drop=True)
N_cal, bin_size = len(cal_sorted), len(cal_sorted) // M_BINS
bins = []
for m in range(M_BINS):
    s  = m * bin_size
    e  = (m + 1) * bin_size if m < M_BINS - 1 else N_cal
    bd = cal_sorted.iloc[s:e]
    bins.append({'m': m, 'l': bd['y_true'].min(), 'u': bd['y_true'].max(), 'data': bd})
    print(f'Bin {m+1}: [{bd["y_true"].min():.1f}, {bd["y_true"].max():.1f}]  N={len(bd)}')

GENDERS = ['F', 'M']
r_hat   = {(m, s): beta for m in range(M_BINS) for s in GENDERS}

def avg_coverage(bins, r_hat, gender):
    covs = []
    for m, b in enumerate(bins):
        g = b['data'][b['data']['gender'] == gender]
        if len(g) == 0:
            continue
        r = r_hat[(m, gender)]
        c = sum(1 for _, row in g.iterrows() if (row['y_lo']-r) <= row['y_true'] <= (row['y_hi']+r))
        covs.append(c / len(g))
    return np.mean(covs) if covs else 0.0

for s in GENDERS:
    print(f'Initial coverage {"Female" if s=="F" else "Male"}: {avg_coverage(bins, r_hat, s):.4f}')

gender
M    30
F    20
Name: count, dtype: int64
Bin 1: [0.0, 3.0]  N=12
Bin 2: [3.0, 12.0]  N=12
Bin 3: [12.0, 21.0]  N=12
Bin 4: [22.0, 43.0]  N=14
Initial coverage Female: 0.9000
Initial coverage Male: 0.9097


In [ ]:
# CELL 16: Fairness-Aware Optimization
TARGET = 1 - ALPHA

def slope_down(bin_data, r, gender):
    g = bin_data[bin_data['gender'] == gender]
    if len(g) == 0: return 0.0
    scores = sorted(g['r'].tolist())
    below  = [s for s in scores if s < r]
    if not below: return 0.0
    return (1.0/len(g)) / (r - below[-1] + 1e-8)

def slope_up(bin_data, r, gender):
    g = bin_data[bin_data['gender'] == gender]
    if len(g) == 0: return float('inf')
    scores = sorted(g['r'].tolist())
    above  = [s for s in scores if s > r]
    if not above: return float('inf')
    return (1.0/len(g)) / (above[0] - r + 1e-8)

print('=== Fairness-Aware Optimization ===')
for iteration in range(500):
    cov   = {s: avg_coverage(bins, r_hat, s) for s in GENDERS}
    if all(abs(cov[s] - TARGET) < 0.02 for s in GENDERS):
        print(f'Converged at iteration {iteration}.')
        break
    over  = max(GENDERS, key=lambda s: cov[s])
    under = min(GENDERS, key=lambda s: cov[s])
    if cov[over] <= TARGET:
        for s in GENDERS:
            for m in range(M_BINS):
                if slope_up(bins[m]['data'], r_hat[(m,s)], s) < float('inf'):
                    r_hat[(m,s)] += 0.1
        continue
    sd_best = max(range(M_BINS), key=lambda m: slope_down(bins[m]['data'], r_hat[(m,over)],  over))
    su_best = min(range(M_BINS), key=lambda m: slope_up(bins[m]['data'],   r_hat[(m,under)], under))
    gd = slope_down(bins[sd_best]['data'], r_hat[(sd_best,over)],  over)
    gu = slope_up(bins[su_best]['data'],   r_hat[(su_best,under)], under)
    if gu > gd and cov[under] >= TARGET - 0.02:
        print(f'Slope condition met at iteration {iteration}.')
        break
    r_hat[(sd_best,over)]  -= 0.1
    r_hat[(su_best,under)] += 0.1
    if iteration % 50 == 0:
        print(f'Iter {iteration:4d} | F: {cov["F"]:.4f}  M: {cov["M"]:.4f}')

final_cov = {s: avg_coverage(bins, r_hat, s) for s in GENDERS}
for s in GENDERS:
    print(f'  {"Female" if s=="F" else "Male":6s}: {final_cov[s]:.4f}  (target {TARGET:.2f})')
print(f'PICP Gap: {abs(final_cov["F"]-final_cov["M"]):.4f}')

=== Fairness-Aware Optimization ===
Converged at iteration 0.
  Female: 0.9000  (target 0.90)
  Male  : 0.9097  (target 0.90)
PICP Gap: 0.0097


In [ ]:
# CELL 17: Apply FUQ to test set
def fuq_interval(y_lo, y_hi, gender, bins, r_hat):
    ulo, uhi = float('inf'), float('-inf')
    for m, b in enumerate(bins):
        r  = r_hat[(m, gender)]
        lo = max(y_lo-r, b['l'])
        hi = min(y_hi+r, b['u'])
        if lo <= hi:
            ulo = min(ulo, lo)
            uhi = max(uhi, hi)
    if ulo == float('inf'):
        r   = r_hat[(0, gender)]
        ulo = y_lo - r
        uhi = y_hi + r
    return ulo, uhi

test_fuq = []
for stem, y_true, q_pred in test_results:
    g      = gender_map.get(stem, 'M')
    lo, hi = fuq_interval(q_pred[Q_LO_IDX], q_pred[Q_HI_IDX], g, bins, r_hat)
    test_fuq.append({'stem': stem, 'y_true': y_true, 'y_pred': q_pred[49],
                     'lo': lo, 'hi': hi, 'gender': g, 'covered': lo<=y_true<=hi})

test_fuq_df = pd.DataFrame(test_fuq)
print('===== FUQ Results =====')
print(f'Overall PICP : {test_fuq_df["covered"].mean():.4f}  (target {1-ALPHA:.2f})')
print(f'Overall MPIW : {(test_fuq_df["hi"]-test_fuq_df["lo"]).mean():.4f}')
picps = {}
for s in GENDERS:
    g = test_fuq_df[test_fuq_df['gender']==s]
    if len(g)==0: continue
    ps = g['covered'].mean()
    picps[s] = ps
    print(f'  {"Female" if s=="F" else "Male":6s} N={len(g):3d}  PICP={ps:.4f}  MPIW={(g["hi"]-g["lo"]).mean():.4f}')
print(f'PICP Gap: {abs(picps.get("F",0)-picps.get("M",0)):.4f}')

===== FUQ Results =====
Overall PICP : 0.8800  (target 0.90)
Overall MPIW : 30.1958
  Female N= 15  PICP=0.8667  MPIW=29.1213
  Male   N= 35  PICP=0.8857  MPIW=30.6563
PICP Gap: 0.0190


In [ ]:
# CELL 18: Comparison table CQR vs FUQ
test_cqr_df = pd.DataFrame([{
    'stem': stem, 'y_true': y, 'lo': lo, 'hi': hi,
    'gender': gender_map.get(stem,'M'), 'covered': lo<=y<=hi
} for stem, y, lo, hi in test_intervals_cqr])

print(f'{"Method":8s} {"PICP":>8s} {"MPIW":>8s} {"PICP(F)":>10s} {"PICP(M)":>10s} {"Gap":>8s}')
print('-'*60)
for name, df in [("CQR", test_cqr_df), ("FUQ", test_fuq_df)]:
    pa = df['covered'].mean()
    mw = (df['hi']-df['lo']).mean()
    pf = df[df['gender']=='F']['covered'].mean() if (df['gender']=='F').any() else float('nan')
    pm = df[df['gender']=='M']['covered'].mean() if (df['gender']=='M').any() else float('nan')
    print(f'{name:8s} {pa:8.4f} {mw:8.4f} {pf:10.4f} {pm:10.4f} {abs(pf-pm):8.4f}')

Method       PICP     MPIW    PICP(F)    PICP(M)      Gap
------------------------------------------------------------
CQR        0.8800  36.5309     0.8667     0.8857   0.0190
FUQ        0.8800  30.1958     0.8667     0.8857   0.0190


In [ ]:
# CELL 19: Age and gender subgroup analysis
test_fuq_df['age_group'] = test_fuq_df['stem'].map(age_map)
test_fuq_df['subgroup']  = test_fuq_df['gender'] + '_' + test_fuq_df['age_group'].fillna('Unknown')

print('=== Test Set Subgroup Distribution ===')
print(test_fuq_df['subgroup'].value_counts())

print('\n=== FUQ Results by Gender x Age Subgroup ===')
print(f'{"Subgroup":20s} {"N":>5s} {"PICP":>8s} {"MPIW":>8s} {"MAE":>8s}')
print('-' * 55)
for sg in ['F_Young', 'F_Old', 'M_Young', 'M_Old']:
    grp = test_fuq_df[test_fuq_df['subgroup'] == sg]
    if len(grp) == 0:
        continue
    mae  = (grp['y_true'] - grp['y_pred']).abs().mean()
    lbl  = sg.replace('F_','Female_').replace('M_','Male_')
    print(f'{lbl:20s} {len(grp):>5d} {grp["covered"].mean():>8.4f} {(grp["hi"]-grp["lo"]).mean():>8.4f} {mae:>8.4f}')

print('\n=== PICP Gaps ===')
picps_sg = {}
for sg in ['F_Young', 'F_Old', 'M_Young', 'M_Old']:
    grp = test_fuq_df[test_fuq_df['subgroup'] == sg]
    if len(grp) > 0:
        picps_sg[sg] = grp['covered'].mean()
for pair in [('F_Young','M_Young'),('F_Old','M_Old'),
             ('F_Young','F_Old'),('M_Young','M_Old'),
             ('F_Young','M_Old'),('F_Old','M_Young')]:
    a, b = pair
    la = a.replace('F_','Female_').replace('M_','Male_')
    lb = b.replace('F_','Female_').replace('M_','Male_')
    gap = abs(picps_sg.get(a,0) - picps_sg.get(b,0))
    print(f'{la:20s} vs {lb:20s} gap: {gap:.4f}')

=== Test Set Subgroup Distribution ===
subgroup
M_Old      23
F_Old      13
M_Young    12
F_Young     2
Name: count, dtype: int64

=== FUQ Results by Gender x Age Subgroup ===
Subgroup                 N     PICP     MPIW      MAE
-------------------------------------------------------
Female_Young             2   1.0000  25.1421   1.3872
Female_Old              13   0.8462  29.7335   9.1198
Male_Young              12   0.8333  28.2328   7.5535
Male_Old                23   0.9130  31.9207   9.1096

=== PICP Gaps ===
Female_Young         vs Male_Young           gap: 0.1667
Female_Old           vs Male_Old             gap: 0.0669
Female_Young         vs Female_Old           gap: 0.1538
Male_Young           vs Male_Old             gap: 0.0797
Female_Young         vs Male_Old             gap: 0.0870
Female_Old           vs Male_Young           gap: 0.0128


In [ ]:
# CELL 20: DeepFace race annotation
# Runs on Colab where DeepFace is compatible
!pip install deepface -q

from deepface import DeepFace
from collections import Counter

RACE_CSV = '/content/drive/MyDrive/avec2014_race.csv'

def annotate_video_race(path, sample_every=30):
    cap   = cv2.VideoCapture(path)
    races = []
    idx   = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % sample_every == 0:
            try:
                result = DeepFace.analyze(
                    frame,
                    actions=['race'],
                    enforce_detection=False,
                    silent=True
                )
                races.append(result[0]['dominant_race'])
            except Exception:
                pass
        idx += 1
    cap.release()
    if not races:
        return 'unknown'
    return Counter(races).most_common(1)[0][0]

all_videos = [(path, stem) for path, stem, _ in train_items + cal_items + test_items]
print(f'Annotating race for {len(all_videos)} videos...')

rows = []
for path, stem in tqdm(all_videos):
    race = annotate_video_race(path)
    rows.append({'filename': stem, 'race': race})

race_df = pd.DataFrame(rows)
print(f'\nRace distribution:')
print(race_df['race'].value_counts())
race_df.to_csv(RACE_CSV, index=False)
print(f'Saved: {RACE_CSV}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 80.3 MB/s eta 0:00:00
.26-06-13 14:59:58 - Directory /root/.deepface has been created
26-06-13 14:59:58 - Directory /root/.deepface/weights has been created
Annotating race for 300 videos...


  0%|          | 0/300 [00:00<?, ?it/s]

26-06-13 14:59:59 - 🔗 race_model_single_batch.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/race_model_single_batch.h5 to /root/.deepface/weights/race_model_single_batch.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/race_model_single_batch.h5
To: /root/.deepface/weights/race_model_single_batch.h5

  0%|          | 0.00/537M [00:00<?, ?B/s]
 11%|█         | 59.8M/537M [00:00<00:00, 594MB/s]
 22%|██▏       | 120M/537M [00:00<00:00, 496MB/s] 
 32%|███▏      | 170M/537M [00:00<00:00, 474MB/s]
 41%|████      | 219M/537M [00:00<00:00, 413MB/s]
 49%|████▊     | 261M/537M [00:00<00:00, 382MB/s]
 56%|█████▌    | 300M/537M [00:00<00:00, 270MB/s]
 62%|██████▏   | 332M/537M [00:01<00:00, 207MB/s]
 66%|██████▋   | 357M/537M [00:01<00:01, 138MB/s]
 70%|███████   | 376M/537M [00:01<00:01, 122MB/s]
 73%|███████▎  | 392M/537M [00:01<00:01, 108MB/s]
 75%|███████▌  | 405M/537M [00:02<00:01, 81.8MB/s]
 77%|███████▋  | 416M/537M [00:02<00:01, 75.3MB/s]
 79%|███████▉  | 426M/537M [00:02<00:01, 78.5MB/s]
 81%|████████  | 435M/537M [00:02<00:01, 77.1MB/s]
 83%|████████▎ | 444M/537M [00:02<00:01, 78.0MB/s]
 84%|████████▍ | 452M/537M [0

.

...

.

 99%|█████████▉| 298/300 [29:37<00:14,  7.12s/it]

.

100%|██████████| 300/300 [30:13<00:00,  6.05s/it]


Race distribution:
race
white              241
latino hispanic     30
asian               18
middle eastern      10
black                1
Name: count, dtype: int64
Saved: /content/drive/MyDrive/avec2014_race.csv


In [ ]:
# CELL 21: Race FUQ analysis
race_map = dict(zip(race_df['filename'], race_df['race']))
test_fuq_df['race'] = test_fuq_df['stem'].map(race_map)

print('=== Test Set Race Distribution ===')
print(test_fuq_df['race'].value_counts())

print('\n=== FUQ Results by Race ===')
print(f'{"Race":25s} {"N":>5s} {"PICP":>8s} {"MPIW":>8s} {"MAE":>8s}')
print('-' * 60)
race_picps = {}
for race in test_fuq_df['race'].dropna().unique():
    grp = test_fuq_df[test_fuq_df['race'] == race]
    if len(grp) == 0:
        continue
    mae  = (grp['y_true'] - grp['y_pred']).abs().mean()
    picp = grp['covered'].mean()
    mpiw = (grp['hi'] - grp['lo']).mean()
    race_picps[race] = picp
    print(f'{race:25s} {len(grp):>5d} {picp:>8.4f} {mpiw:>8.4f} {mae:>8.4f}')

print('\n=== PICP Gaps Between Race Groups ===')
races = list(race_picps.keys())
for i, r1 in enumerate(races):
    for r2 in races[i+1:]:
        gap = abs(race_picps[r1] - race_picps[r2])
        print(f'{r1:25s} vs {r2:25s} gap: {gap:.4f}')

=== Test Set Race Distribution ===
race
white              42
latino hispanic     6
asian               2
Name: count, dtype: int64

=== FUQ Results by Race ===
Race                          N     PICP     MPIW      MAE
------------------------------------------------------------
white                        42   0.9048  29.8608   8.3798
latino hispanic               6   0.6667  33.9462  11.3410
asian                         2   1.0000  25.9786   0.7479

=== PICP Gaps Between Race Groups ===
white                     vs latino hispanic           gap: 0.2381
white                     vs asian                     gap: 0.0952
latino hispanic           vs asian                     gap: 0.3333


In [ ]:
# CELL 22: Save all results to Drive
test_fuq_df.to_csv('/content/drive/MyDrive/avec2014_fuq_results_full.csv', index=False)
print('Full results saved to Drive.')

print('\n===== Complete Summary =====')
print(f'QR  Test MAE  : {test_mae:.4f}')
print(f'CQR PICP      : {test_cqr_df["covered"].mean():.4f}')
print(f'CQR MPIW      : {(test_cqr_df["hi"]-test_cqr_df["lo"]).mean():.4f}')
cqr_f = test_cqr_df[test_cqr_df.gender=='F']['covered'].mean()
cqr_m = test_cqr_df[test_cqr_df.gender=='M']['covered'].mean()
fuq_f = test_fuq_df[test_fuq_df.gender=='F']['covered'].mean()
fuq_m = test_fuq_df[test_fuq_df.gender=='M']['covered'].mean()
print(f'CQR PICP Gap  : {abs(cqr_f-cqr_m):.4f}')
print(f'FUQ PICP      : {test_fuq_df["covered"].mean():.4f}')
print(f'FUQ MPIW      : {(test_fuq_df["hi"]-test_fuq_df["lo"]).mean():.4f}')
print(f'FUQ PICP Gap  : {abs(fuq_f-fuq_m):.4f}')

Full results saved to Drive.

===== Complete Summary =====
QR  Test MAE  : 8.4299
CQR PICP      : 0.8800
CQR MPIW      : 36.5309
CQR PICP Gap  : 0.0190
FUQ PICP      : 0.8800
FUQ MPIW      : 30.1958
FUQ PICP Gap  : 0.0190
